<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

Welcome back! Last week we played with the **Spotify dataset** and got a taste of Hugging Face pipelines.
Today we go further: we'll turn raw song data into **features**, add a fun **AI-generated feature** using a
pretrained language model, and then train and compare **6 different models** to predict whether a song is a hit.

By the end of this notebook you will be able to:
1. Engineer simple, useful features from raw data
2. Use a pretrained Hugging Face model to auto-generate a new feature (no training needed!)
3. Turn text/categories into numbers (One-Hot Encoding vs Label Encoding)
4. Split data the *correct* way for classification (Stratified Split)
5. Train 6 classic ML models + 1 Deep Learning model
6. Evaluate models properly (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix)
7. Decide which mistake (False Positive or False Negative) matters more in real life
8. Do error analysis like a real ML engineer

Runs fully on **Google Colab (free, GPU)**


## 0. Setup

We only need a handful of libraries. `transformers` is the only "heavy" one, but the model we use is tiny
(~250MB, distilled) and runs on CPU in a few seconds per batch.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded ✅")


## 1. Load the Spotify Dataset (our old friend 👋)

Same dataset as before — ~114k Spotify tracks with audio features like `danceability`, `energy`, `tempo`, etc.


In [ ]:
# ==========================
# TODO: Student: Enter your dataset name
# ==========================
YOUR_DATASET_NAME = ""

url = f"https://raw.githubusercontent.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/main/dataset/{YOUR_DATASET_NAME}.csv"
df = pd.read_csv(url, index_col=0)
print(df.shape)
df.head()


In [ ]:
df.info()


## 2. Quick Recap — Feature Engineering (5 easy wins)

**Feature Engineering** = creating *new* columns from existing ones so that a model can learn patterns more easily.
A good feature engineer often improves a model more than switching to a fancier algorithm!

We'll add 5 simple, high-value features:

| # | New Feature | Why it helps |
|---|---|---|
| 1 | `duration_min` | Minutes are more intuitive than milliseconds |
| 2 | `is_explicit` | Convert `True/False` text to `0/1` so models can use it directly |
| 3 | `energy_dance_ratio` | Captures the *interaction* between energy and danceability — a single number often beats two separate ones |
| 4 | `tempo_bucket` | Groups tempo into "slow / medium / fast" — sometimes coarse buckets generalize better than raw numbers |
| 5 | `loudness_norm` | Rescales loudness (which is negative dB, e.g. -60 to 0) into a friendlier 0-1 range |


In [ ]:
# 1. Duration in minutes (easier to read than milliseconds)
df["duration_min"] = # Write your code here

# 2. Explicit flag as 0/1 instead of True/False
df["is_explicit"] = # Write your code here

# 3. Interaction feature: energy vs danceability ratio
df["energy_dance_ratio"] = # Write your code here

# 4. Tempo bucket (simple binning -> a NEW categorical feature)
df["tempo_bucket"] = pd.cut(
    df["tempo"],
    bins=[0, 90, 120, 1000],
    labels=["slow", "medium", "fast"]
)

# 5. Normalize loudness into a 0-1 range (min-max scaling on a single column)
df["loudness_norm"] = (df["loudness"] - df["loudness"].min()) / (df["loudness"].max() - df["loudness"].min())

df[["duration_min", "is_explicit", "energy_dance_ratio", "tempo_bucket", "loudness_norm"]].head()


> 💡 **Your turn!** Before looking at my solution, think of **one new feature** that you would engineer from the existing data. Discuss it with your classmates and explain why it might be useful.
>
> Common ideas include:
>
> * `valence * energy` (*hype score*)
> * `speechiness > 0.66` (mostly spoken content?)
> * Average popularity for each genre


## 3. 🤖 A "Modern AI" Feature — Zero-Shot Mood Detection

Last week we ran out of time on Hugging Face, so let's use it *today* to create a genuinely new feature —
without training anything ourselves!

**Zero-shot classification** means: the model was never trained on our specific labels (`happy`, `sad`,
`romantic`, ...), yet it can still guess which one fits best, just from understanding language. It's a great
first taste of *transfer learning* and "modern AI" feature engineering — using a big pretrained model to
generate a feature for a small downstream ML model.

We'll only run it on a **small sample** (~40 tracks) so it finishes in well under a minute on Colab's free CPU.


In [ ]:
from transformers import pipeline

# Load a small, CPU-friendly zero-shot classification model (~250MB)
zero_shot = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1,  # -1 = force CPU
)

candidate_moods = # Write your code here
print("Model loaded ✅")


In [ ]:
# Take a small sample so this stays fast on free Colab CPU
sample = df.sample(40, random_state=RANDOM_STATE).copy()

moods, mood_scores = [], []
for _, row in sample.iterrows():
    text = f"{row['track_name']} by {row['artists']}"
    result = zero_shot(text, candidate_moods)
    moods.append(result["labels"][0])          # top predicted mood
    mood_scores.append(result["scores"][0])     # model's confidence

sample["ai_mood"] = moods
sample["ai_mood_confidence"] = mood_scores

sample[["track_name", "artists", "ai_mood", "ai_mood_confidence"]].head(10)


Notice: the model is reading **only the song/artist name text**, not the audio! Sometimes it nails it,
sometimes it's funny/wrong — that's a great class discussion: *"When should we trust a zero-shot label enough
to use it as a feature?"*

> 🎓 **Teaching moment:** this is the difference between **hand-crafted features** (Section 2, features we
> defined with formulas) and **AI-generated features** (Section 3, a feature *learned* by a huge pretrained
> model). We'll come back to this "manual vs automatic feature extraction" idea again with Deep Learning later!

For the rest of the notebook we'll go back to using the **full dataset** with our Section 2 hand-crafted
features (the zero-shot feature only ran on 40 rows — running it on all ~114k rows would take too long for a
live class, but you now know exactly how to scale it up if you wanted to).


## 4. Turning Text into Numbers: One-Hot vs Label Encoding

Machine learning models only understand **numbers**. So how do we handle a column like `track_genre`
(acoustic, afrobeat, pop, rock, ...) or `tempo_bucket` (slow, medium, fast)?

There are two common approaches:

### 🔢 Option A — Label / Ordinal Encoding
Just assign each category an integer: `slow=0, medium=1, fast=2`.

- ✅ Simple, keeps only 1 column
- ⚠️ Only makes sense when categories have a **real order** (small < medium < large). Our `tempo_bucket` does!
- ❌ **Dangerous** for categories with **no order** (like genres) — the model might wrongly assume
  `"pop" (label=5) > "acoustic" (label=1)`, which is meaningless.

### 🎯 Option B — One-Hot Encoding
Create one new 0/1 column *per category*. E.g. `genre_pop`, `genre_rock`, `genre_acoustic`, ...

- ✅ No fake ordering is introduced — safe for unordered categories like `track_genre`
- ⚠️ Can create **many** columns if there are many categories (our dataset has 114 genres!)

**Rule of thumb:** ordered categories → Label Encoding. Unordered categories → One-Hot Encoding
(or group rare categories together first if there are too many).


In [ ]:
# --- Option A: Label/Ordinal Encoding for tempo_bucket (it HAS a natural order: slow < medium < fast) ---
tempo_order = {"slow": 0, "medium": 1, "fast": 2}
df["tempo_bucket_encoded"] = df["tempo_bucket"].map(tempo_order)

df[["tempo", "tempo_bucket", "tempo_bucket_encoded"]].head()


In [ ]:
# --- Option B: One-Hot Encoding for track_genre (NO natural order -> one-hot is the safe choice) ---
# To keep things manageable for class, let's demo one-hot on a SMALL subset of genres first
demo_genres = df["track_genre"].value_counts().head(5).index.tolist()
demo = df[df["track_genre"].isin(demo_genres)][["track_genre"]].copy()

demo_onehot = pd.get_dummies(demo["track_genre"], prefix="genre")
pd.concat([demo, demo_onehot], axis=1).drop_duplicates().reset_index(drop=True)


👀 See how each genre becomes its own 0/1 column, with no implied ranking between them? That's the safe
choice for unordered categories. We'll use this idea (in a lighter form) when we build our final feature set
below.


## 5. Defining Our Supervised Learning Problem

**Supervised learning** = we have inputs (`X`, our features) and a known correct answer (`y`, the target),
and we want the model to learn the mapping `X -> y`.

**Today's task (classification):** predict whether a track is a **Hit** or **Not a Hit**. This turns a numeric
column (`popularity`) into a binary classification target — a very common real-world pattern (e.g. "will this
customer churn?", "is this transaction fraud?").

⚠️ **A trap to avoid:** picking a fixed cutoff like `popularity >= 70` sounds reasonable, but on this dataset
it labels only ~5% of tracks as "Hit" — severe imbalance. A model can then score ~95% accuracy just by *always*
predicting "Not a Hit", and F1 on the minority class collapses. This is a real, common mistake, not just a class
demo problem!

**The fix:** define "Hit" using a **percentile** of the data instead of a fixed number — e.g. the top 25% most
popular tracks. This is a much more robust way to define a target for classification when a domain-specific
cutoff isn't obvious, and it gives us a reasonable class balance to work with.


In [ ]:
# Percentile-based threshold instead of a fixed number -> much healthier class balance
# top 25% most popular tracks = "Hit"
HIT_PERCENTILE = # Write your code here
threshold_value = df["popularity"].quantile(HIT_PERCENTILE)

df["is_hit"] = (df["popularity"] >= threshold_value).astype(int)

print(f"Popularity cutoff for 'Hit' (top {int((1-HIT_PERCENTILE)*100)}%): {threshold_value}")
print(df["is_hit"].value_counts(normalize=True))
sns.countplot(x="is_hit", data=df)
plt.title("Class Balance: Hit (1) vs Not a Hit (0) — percentile-based threshold")
plt.show()


⚠️ Notice the classes are **imbalanced** (hits are rarer than non-hits). This is exactly why we must use
**stratified splitting** next — otherwise our test set could end up with almost no hits at all, and our
evaluation would be misleading.


## 6. Before We Touch Any Model — The Pre-Modeling Checklist ✅

A few things to always do *before* training, or your results can be silently wrong:

1. **Handle missing values** — models (mostly) can't handle `NaN`.
2. **Remove target leakage** — never let a feature that "gives away" the answer sneak into `X`
   (e.g. `popularity` itself must NOT be a feature, since we derived `is_hit` directly from it!).
3. **Encode categorical features** — we just learned One-Hot vs Label encoding above.
4. **Scale numeric features** — some models (Logistic Regression, SVM, Neural Networks) are sensitive to
   feature scale; tree-based models (Decision Tree, Random Forest, XGBoost) don't need scaling, but it never hurts.
5. **Split BEFORE fitting anything** (scaler, encoder, model) using the **training set only**, to avoid leaking
   test-set information — and use a **stratified split** for classification with imbalanced classes.


In [ ]:
# Build our final feature set
feature_cols = [
    "danceability", "energy", "loudness_norm", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_min",
    "is_explicit", "energy_dance_ratio", "tempo_bucket_encoded"
]

model_df = df.dropna(subset=feature_cols + ["is_hit"]).copy()

X = # Write your code here
y = # Write your code here

print("Missing values in X:", X.isna().sum().sum())
print("Shape:", X.shape)


In [ ]:
# STRATIFIED train/test split -- keeps the same Hit/Not-Hit ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    # Write your code here
)

print("Train hit-rate:", y_train.mean().round(3))
print("Test hit-rate :", y_test.mean().round(3))

# Scale AFTER splitting, fit ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaling done ✅ (fit on train only, applied to both)")


## 7. Training 6 Classic Models — With Hand-Crafted Features

Here's the quick intuition for each model (great for a whiteboard recap):

| Model | Intuition in one line |
|---|---|
| **Logistic Regression** | Draws a straight (linear) decision boundary based on a weighted sum of features |
| **Decision Tree** | Asks a series of yes/no questions ("is danceability > 0.6?") to reach a decision |
| **Random Forest** | Trains *many* Decision Trees on random subsets and lets them vote — reduces overfitting |
| **XGBoost** | Trains trees *sequentially*, each new tree fixing the mistakes of the previous ones (gradient boosting) |
| **SVM** | Finds the boundary that maximizes the *margin* (gap) between the two classes |
| **Neural Network (MLP)** | Stacks layers of weighted connections + non-linear activations to learn complex patterns |

We'll train all 6 with the **same** train/test split so the comparison is fair.

⚖️ **Handling the remaining imbalance:** even with our percentile-based target, the classes aren't perfectly
50/50. Several sklearn models accept `class_weight="balanced"`, which automatically up-weights the minority
class during training (mistakes on "Hit" tracks are penalized more) instead of letting the model coast by
mostly predicting the majority class. XGBoost has an equivalent knob, `scale_pos_weight`.


In [ ]:
# scale_pos_weight tells XGBoost how much more to weight the minority (Hit) class
pos_weight =# Write your code here

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, eval_metric="logloss", scale_pos_weight=pos_weight, random_state=RANDOM_STATE, n_jobs=-1),
    # "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    # "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=RANDOM_STATE),
    # Note: MLPClassifier has no built-in class_weight -- a great discussion point!
    # (workaround: manually oversample the minority class, or use sample_weight where supported)
}

results = {}   # will store metrics for every model, ML and DL alike

# tqdm gives us a live progress bar -- great for a class demo when SVM/MLP take a while on free CPU
progress_bar = tqdm(models.items(), total=len(models), desc="Training models")

for name, model in progress_bar:
    progress_bar.set_postfix_str(f"now training: {name}")

    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    train_seconds = time.time() - start_time

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        "y_pred": y_pred,
        "y_proba": y_proba,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba),
        "train_seconds": train_seconds,
    }

    # Update the bar's postfix so students see accuracy + time for the model that JUST finished
    progress_bar.set_postfix_str(
        f"{name}: Acc={results[name]['accuracy']:.3f}  ({train_seconds:.1f}s)"
    )
    tqdm.write(
        f"{name:22s} | Acc={results[name]['accuracy']:.3f}  F1={results[name]['f1']:.3f}  "
        f"AUC={results[name]['auc']:.3f}  |  Train time: {train_seconds:.2f}s"
    )


In [ ]:
# Retrieve the trained Logistic Regression model
lr_model = # Write your code here

# Create a DataFrame containing each feature and its coefficient
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_model.coef_[0],
    "Abs_Coefficient": np.abs(lr_model.coef_[0])  # Absolute value indicates overall importance
})

# Sort features by the magnitude of their coefficients
coef_df = coef_df.sort_values("Abs_Coefficient", ascending=False)

# Display the ranked feature importance table
print(coef_df)

In [ ]:
import matplotlib.pyplot as plt

# Sort by coefficient value so negative and positive effects are easy to see
plot_df = coef_df.sort_values("Coefficient")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["Feature"], plot_df["Coefficient"])

# Draw a vertical reference line at zero
plt.axvline(0, color="black", linestyle="--")

plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Feature Importance (Logistic Regression)")

plt.tight_layout()
plt.show()

### ⏱️ Training Time Leaderboard

Which model was the slowest to train? Usually **SVM** on non-linear kernels scales poorly with more rows, and
**Random Forest** / **XGBoost** take longer than simple **Logistic Regression** because they build many trees.
This is a good moment to talk about the **accuracy vs training-time trade-off** — the "best" model on paper
isn't always worth using if it's 50x slower for only a tiny accuracy gain.


In [ ]:
timing_summary = pd.DataFrame({
    name: {"train_seconds": r["train_seconds"], "accuracy": r["accuracy"], "f1": r["f1"]}
    for name, r in results.items()
}).T.sort_values("train_seconds", ascending=False)

display(timing_summary.round(3))

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.bar(timing_summary.index, timing_summary["train_seconds"], color="steelblue", alpha=0.8, label="Train time (s)")
ax1.set_ylabel("Training time (seconds)", color="steelblue")
ax1.tick_params(axis="x", rotation=30)

ax2 = ax1.twinx()
ax2.plot(timing_summary.index, timing_summary["accuracy"], color="darkorange", marker="o", label="Accuracy")
ax2.set_ylabel("Accuracy", color="darkorange")
ax2.set_ylim(0, 1)

plt.title("Training Time vs Accuracy per Model")
fig.tight_layout()
plt.show()

slowest = timing_summary.index[0]
fastest = timing_summary.index[-1]
print(f"🐌 Slowest to train: {slowest} ({timing_summary.loc[slowest, 'train_seconds']:.2f}s)")
print(f"⚡ Fastest to train: {fastest} ({timing_summary.loc[fastest, 'train_seconds']:.2f}s)")


## 8. Manual Feature Engineering vs "Let the AI Find Its Own Features" (1D-CNN)

So far, every model above only sees the **hand-crafted features** *we* designed in Section 2 — this is the
classic Machine Learning workflow: a human decides what matters.

**Deep Learning** offers a different philosophy: give the network the *raw* numeric feature vector and let it
learn its own internal features through layers, instead of us engineering them by hand.

### 🧠 Quick Conv1D crash course
- A **Convolution (Conv) layer** slides a small "window" (kernel) across the input and learns a filter that
  detects local patterns — think of it as the network *inventing its own mini hand-crafted features*, but
  learned automatically from data instead of designed by a human.
- **Pooling** shrinks the output by keeping only the strongest signal in each region (reduces size, keeps
  the important parts).
- **Dense (fully connected) layers** at the end combine everything the conv layers found into a final decision.

For tabular data like ours, a 1D-CNN treats our 13 features as a short "sequence" and slides a small filter
across neighboring features — a nice, fast way to demonstrate the *concept* of automatic feature extraction,
even though CNNs really shine on images/audio/sequences with genuine spatial structure.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(RANDOM_STATE)

n_features = # Write your code here

# Reshape for Conv1D: (samples, timesteps, channels)
X_train_cnn = X_train_scaled.reshape(-1, n_features, 1)
X_test_cnn = X_test_scaled.reshape(-1, n_features, 1)

cnn = models.Sequential([
    layers.Input(shape=(n_features, 1)),
    layers.Conv1D(filters=16, kernel_size=3, activation="relu"),   # learns its own local feature detectors
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),   # binary classification output
])

cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn.summary()


In [ ]:
# Keras equivalent of class_weight="balanced": weight the minority class more in the loss function
cnn_class_weight = {0: 1.0, 1: float(pos_weight)}

from tqdm.keras import TqdmCallback   # gives us a live per-epoch progress bar, just like the ML models above

cnn_start_time = time.time()
history = cnn.fit(
    X_train_cnn, y_train,
    validation_split=0.15,
    epochs=15,
    batch_size=64,
    class_weight=cnn_class_weight,
    verbose=0,
    callbacks=[TqdmCallback(verbose=1, desc="Training 1D-CNN")],
)
cnn_train_seconds = time.time() - cnn_start_time

y_proba_cnn = cnn.predict(X_test_cnn, verbose=0).ravel()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)

results["1D-CNN (Deep Learning)"] = {
    "y_pred": y_pred_cnn,
    "y_proba": y_proba_cnn,
    "accuracy": accuracy_score(y_test, y_pred_cnn),
    "precision": precision_score(y_test, y_pred_cnn, zero_division=0),
    "recall": recall_score(y_test, y_pred_cnn, zero_division=0),
    "f1": f1_score(y_test, y_pred_cnn, zero_division=0),
    "auc": roc_auc_score(y_test, y_proba_cnn),
    "train_seconds": cnn_train_seconds,
}
print(
    f"1D-CNN trained ✅  Test accuracy: {results['1D-CNN (Deep Learning)']['accuracy']:.3f}  "
    f"|  Train time: {cnn_train_seconds:.2f}s"
)


In [ ]:
# Training curves - a good habit to always check for overfitting
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()


🎓 **Discussion for class:** did the CNN beat the hand-crafted-feature models? With only 13 tabular
features and a few thousand rows, classic ML (especially boosted trees) very often **wins or ties** deep
learning — deep learning's advantage usually shows up with much larger data, or raw signals like images/audio/text
where hand-crafting features is hard or impossible. That's an important, realistic lesson: **bigger/fancier
isn't automatically better — it depends on your data.**


## 9. Comparing All 7 Models Side by Side

🎓 **Full-circle moment:** this is exactly why we spent time in Section 5 fixing the target definition and in
Section 7 adding `class_weight`. If we'd kept the original `popularity >= 70` cutoff (~5% positive class) and
trained without class weighting, most models would have learned to almost always predict "Not a Hit", giving
high **Accuracy** but a **terrible F1 / Recall on the Hit class** — a classic imbalanced-classification trap.
Compare the F1 column below to Accuracy and you'll see they now tell a much more consistent story.


In [ ]:
summary = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ("y_pred", "y_proba")}
    for name, r in results.items()
}).T.sort_values("f1", ascending=False)

summary.style.background_gradient(cmap="Greens", axis=0)


In [ ]:
summary[["accuracy", "precision", "recall", "f1", "auc"]].plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1)
)
plt.title("Model Comparison Across Metrics")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 10. Evaluation Deep Dive — Confusion Matrix & Metrics

Let's pick our best model (by F1) and really understand what "good" means.


In [ ]:
best_model_name = # Write your code here
print("Best model by F1:", best_model_name)

y_pred_best = results[best_model_name]["y_pred"]
cm = confusion_matrix(y_test, y_pred_best)

ConfusionMatrixDisplay(cm, display_labels=["Not Hit (0)", "Hit (1)"]).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.show()


### The 4 building blocks

|  | Predicted: Not Hit (0) | Predicted: Hit (1) |
|---|---|---|
| **Actual: Not Hit (0)** | ✅ True Negative (TN) | ❌ False Positive (FP) |
| **Actual: Hit (1)** | ❌ False Negative (FN) | ✅ True Positive (TP) |

- **TP (True Positive):** model said "Hit", it really was a Hit
- **TN (True Negative):** model said "Not Hit", it really wasn't
- **FP (False Positive):** model said "Hit", but it wasn't (a **false alarm**)
- **FN (False Negative):** model said "Not Hit", but it actually was a Hit (a **missed catch**)

### The formulas

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \quad\text{— overall, how often are we correct?}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad\text{— of everything we called "Hit", how many really were?}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad\text{— of all the actual Hits, how many did we catch?}$$

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad\text{— balances Precision and Recall into one number}$$

$$\text{AUC (ROC)} = \text{how well the model ranks positives above negatives, across every possible threshold}$$


In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print()
print(f"Accuracy  = (TP+TN)/(TP+TN+FP+FN) = {(tp+tn)/(tp+tn+fp+fn):.3f}")
print(f"Precision = TP/(TP+FP)            = {tp/(tp+fp):.3f}")
print(f"Recall    = TP/(TP+FN)            = {tp/(tp+fn):.3f}")
print(f"F1        = 2*P*R/(P+R)           = {2*(tp/(tp+fp))*(tp/(tp+fn))/((tp/(tp+fp))+(tp/(tp+fn))):.3f}")


### 🏥 Which mistake matters more, FP or FN? (Medical example)

Imagine the model is screening patients for a serious disease (Positive = "has disease"):

- A **False Negative (FN)** means telling a sick patient "you're fine" — they go home undiagnosed and untreated.
  This can be **fatal**.
- A **False Positive (FP)** means telling a healthy patient "you might be sick" — scary and inconvenient, but
  usually just leads to a follow-up test.

👉 In medical screening, we almost always care **far more about minimizing False Negatives**, i.e. we want
**high Recall**, even if it costs us some Precision (more false alarms are an acceptable trade-off for not
missing real cases).

### 💼 What about business problems?

It depends entirely on what a False Positive vs False Negative *costs* the business:

- **Spam detection:** a False Positive (real email marked as spam) can mean a missed job offer or important
  contract — often worse than letting a little spam through. Businesses often prioritize **Precision** here.
- **Fraud detection:** a False Negative (real fraud missed) can cost real money directly — so **Recall**
  tends to matter more, similar to the medical case.
- **Marketing / churn prediction:** a False Positive (offering a discount to a customer who wasn't going to
  churn anyway) just costs a bit of margin — usually a cheap mistake, so teams can tolerate lower Precision to
  get higher Recall and not lose real at-risk customers.

👉 **There's no universal right answer** — a good ML engineer always asks: *"What does each type of mistake
cost us, in real terms?"* before choosing which metric to optimize.


## 11. ROC Curves for All Models


In [ ]:
plt.figure(figsize=(7, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['auc']:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curves — All Models")
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()


## 12. Error Analysis — What is our best model getting *wrong*?

Looking at the actual misclassified rows is one of the most valuable habits in ML — it often reveals data
quality issues or missing features, not just "the model needs to be bigger".


In [ ]:
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = y_pred_best
error_df["track_name"] = model_df.loc[X_test.index, "track_name"].values
error_df["artists"] = model_df.loc[X_test.index, "artists"].values

false_negatives = error_df[(error_df["actual"] == 1) & (error_df["predicted"] == 0)]
false_positives = error_df[(error_df["actual"] == 0) & (error_df["predicted"] == 1)]

print(f"False Negatives (missed hits): {len(false_negatives)}")
print(f"False Positives (false alarms): {len(false_positives)}")

print("\nSample missed hits:")
false_negatives[["track_name", "artists", "danceability", "energy", "valence"]].head(5)


In [ ]:
# Compare feature distributions: correct vs incorrect predictions
error_df["correct"] = error_df["actual"] == error_df["predicted"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, ["danceability", "energy", "valence"]):
    sns.boxplot(x="correct", y=feat, data=error_df, ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.show()


💬 **Discussion:** do misclassified tracks look systematically different (e.g. mid-range danceability,
ambiguous valence)? This is exactly the kind of insight that tells us **which new feature to engineer next** —
bringing us right back to Section 2!


## 13. Wrap-Up — What We Learned Today 🎉

1. **Feature engineering** turns raw columns into more useful signals (Section 2)
2. Pretrained models can generate **new AI-powered features** with zero training (Section 3, Hugging Face)
3. **One-Hot vs Label Encoding** — pick based on whether categories have a natural order (Section 4)
4. Always **split before scaling/fitting**, and use **stratified splits** for imbalanced classification (Section 6)
5. We trained **6 classic ML models + 1 Deep Learning model** and compared them fairly (Sections 7-8)
6. **Accuracy alone can lie** — Precision, Recall, F1, and AUC each tell a different part of the story (Section 10)
7. Whether **False Positives or False Negatives** matter more depends entirely on real-world cost (medical vs
   business examples)
8. **Error analysis** turns "the model is wrong" into "here's what to fix next"

### Final leaderboard


In [ ]:
summary[["accuracy", "precision", "recall", "f1", "auc"]].round(3)


---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 3 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>